In [ ]:
# Ref: https://huggingface.co/blog/zh/welcome-openai-gpt-oss
# requires: torch 2.8, triton 3.4
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
    # Flash Attention with Sinks
    # attn_implementation="kernels-community/vllm-flash-attn3",

    # Optimize MoE layers with downloadable` MegaBlocksMoeMLP
    # use_kernels=True,
)

messages = [
    {"role": "user", "content": "How many rs are in the word 'strawberry'?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))
